EXERCICE GOLD — Extractive Summarization

In [34]:
# Cellule 16 — Installation
!pip install nltk numpy --quiet

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

import numpy as np
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
import string

print("Setup Gold terminé !")

Setup Gold terminé !


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [35]:
# Cellule 17 — Télécharger GloVe (vecteurs de mots pré-entraînés)
# Simuler avec Sentence Transformers si GloVe n'est pas disponible

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_glove = SentenceTransformer('all-MiniLM-L6-v2')

def vectoriser_phrase(phrase):
    """Convertit une phrase en vecteur"""
    return model_glove.encode([phrase])[0]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [36]:
# Cellule 18 — Nettoyage du texte
stop_words = set(stopwords.words('english'))

def nettoyer_phrase(phrase):
    """Nettoyage basique d'une phrase"""
    mots = word_tokenize(phrase.lower())
    mots_filtres = [
        mot for mot in mots
        if mot not in stop_words
        and mot not in string.punctuation
    ]
    return " ".join(mots_filtres)

In [37]:
# Cellule 19 — Matrice de similarité cosinus + PageRank
def pagerank_simple(matrice_similarite, d=0.85, iterations=100):
    """
    Algorithme PageRank simplifié.

    d         : facteur d'amortissement (0.85 = valeur standard Google)
    iterations : nombre d'itérations jusqu'à convergence

    Retourne un score d'importance pour chaque phrase.
    """
    n = matrice_similarite.shape[0]
    scores = np.ones(n) / n  # initialiser tous les scores à égalité

    for _ in range(iterations):
        nouveaux_scores = np.zeros(n)
        for i in range(n):
            for j in range(n):
                if i != j and matrice_similarite[j].sum() > 0:
                    nouveaux_scores[i] += (
                        d * matrice_similarite[j][i] /
                        matrice_similarite[j].sum() * scores[j]
                    )
            nouveaux_scores[i] += (1 - d) / n
        scores = nouveaux_scores

    return scores


def summarize_article(texte, top_n=3):
    """
    Résume un article en sélectionnant les top_n phrases les plus importantes.

    texte  : article complet
    top_n  : nombre de phrases à retourner
    """
    # 1. Découper en phrases
    phrases = sent_tokenize(texte)
    if len(phrases) <= top_n:
        return texte

    # 2. Vectoriser chaque phrase
    vecteurs = np.array([vectoriser_phrase(p) for p in phrases])

    # 3. Calculer la matrice de similarité cosinus
    matrice_sim = cosine_similarity(vecteurs)

    # 4. Appliquer PageRank
    scores = pagerank_simple(matrice_sim)

    # 5. Sélectionner les top_n phrases
    indices_top = scores.argsort()[-top_n:][::-1]
    indices_top = sorted(indices_top)  # garder l'ordre original

    resume = " ".join([phrases[i] for i in indices_top])
    return resume


# Test avec un article de tennis
article_tennis = """
Rafael Nadal won the French Open for the 14th time in his career.
He defeated Casper Ruud in the final at Roland Garros.
The match lasted three sets and was completed in just over two hours.
Nadal demonstrated his typical clay court dominance throughout the tournament.
His performance was exceptional, winning every set in the tournament.
The crowd at Roland Garros gave him a standing ovation.
This victory cemented his status as the greatest clay court player of all time.
He will now prepare for Wimbledon.
"""

resume = summarize_article(article_tennis, top_n=3)
print("=== ARTICLE ORIGINAL ===")
print(article_tennis)
print("\n=== RÉSUMÉ (3 phrases) ===")
print(resume)

=== ARTICLE ORIGINAL ===

Rafael Nadal won the French Open for the 14th time in his career.
He defeated Casper Ruud in the final at Roland Garros.
The match lasted three sets and was completed in just over two hours.
Nadal demonstrated his typical clay court dominance throughout the tournament.
His performance was exceptional, winning every set in the tournament.
The crowd at Roland Garros gave him a standing ovation.
This victory cemented his status as the greatest clay court player of all time.
He will now prepare for Wimbledon.


=== RÉSUMÉ (3 phrases) ===
Nadal demonstrated his typical clay court dominance throughout the tournament. His performance was exceptional, winning every set in the tournament. This victory cemented his status as the greatest clay court player of all time.
